# 第7回：数値を予測する—回帰

**今日の問い：連続値の予測モデルを、何と比べればよいか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- MAE・RMSE・R²を異なる視点として読む
- Dummyと複数モデルを同条件で比較する
- 残差を群別に調べて次の仮説を作る

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

### 先に押さえる言葉

- MAE：絶対誤差の平均
- RMSE：大きな誤差をより重く扱う指標
- R²：平均予測と比べた当てはまり
- 残差：実測値と予測値の差

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for candidate in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP"]:
    if candidate in available_fonts:
        plt.rcParams["font.family"] = candidate
        break

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["yield_pct"], test_size=0.25, random_state=42)


## TRY：4モデルを同じ条件で比べる


In [ ]:
models = {
    "平均値": DummyRegressor(),
    "線形回帰": LinearRegression(),
    "決定木": DecisionTreeRegressor(max_depth=4, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=6, random_state=42),
}
results, predictions = [], {}
for name, estimator in models.items():
    pipeline = make_pipeline(SimpleImputer(strategy="median"), estimator).fit(X_train, y_train)
    pred = pipeline.predict(X_valid)
    predictions[name] = pred
    results.append({"モデル": name, "MAE": mean_absolute_error(y_valid, pred), "RMSE": mean_squared_error(y_valid, pred) ** 0.5, "R2": r2_score(y_valid, pred)})
pd.DataFrame(results).sort_values("MAE").round(3)


## 予測と実測、残差を見る


In [ ]:
pred = predictions["Random Forest"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_valid, pred, alpha=0.65)
axes[0].plot([y_valid.min(), y_valid.max()], [y_valid.min(), y_valid.max()], "--")
axes[0].set(xlabel="実測収率", ylabel="予測収率", title="予測と実測")
axes[1].scatter(pred, y_valid - pred, alpha=0.65)
axes[1].axhline(0, linestyle="--")
axes[1].set(xlabel="予測収率", ylabel="残差（実測-予測）", title="残差")
plt.tight_layout()


In [ ]:
errors = df.loc[y_valid.index, ["sample_id", "scaffold_group", "catalyst", "yield_pct"]].copy()
errors["予測"] = pred
errors["絶対誤差"] = abs(errors["yield_pct"] - errors["予測"])
errors.nlargest(8, "絶対誤差").round(2)


## CHANGE

`max_depth=6`を`3`または`10`へ変えます。MAEだけでなく、残差図と大きく外した試料も比較します。


## DEEP DIVE：結果を一段深く読む

次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

### 出力を見る観点

- MAEは目的変数と同じ単位で説明できる
- 全体指標が良くても特定系列で外すことがある
- 残差の模様は未学習の構造を示すことがある


In [ ]:
errors["残差"] = errors["yield_pct"] - errors["予測"]
group_error = errors.groupby("scaffold_group").agg(件数=("残差", "size"), MAE=("絶対誤差", "mean"), 平均残差=("残差", "mean"))
display(group_error.sort_values("MAE", ascending=False).round(2))
print("平均残差が正なら、その系列を平均的に過小予測しています。")


## よくある誤り

- R²だけで利用可能と判断する
- テストデータでモデルを選ぶ
- 大きな誤差を外れ値としてすぐ除く

## SELF-STUDY（任意・30〜60分）

- 触媒別と系列別のMAEを計算する
- MAE 5ポイントが業務上許容できるか利用場面から考える

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. MAEとRMSEは何を違って重視するか
2. Dummyより悪い場合に何を見直すか
3. 残差図に模様があると何を疑うか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
